In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# ۱. خواندن فایل اصلی
file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_generator_bearings_deviation_monitoring\deviation_monitoring\dsas_g11_generator_bearings_deviation_monitoring_output5.xlsx'

# لیست فیچرها و تارگت‌ها
all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
# لیست سنسورهایی که تارگت هستند
target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
# ۲. بارگذاری و پیش‌پردازش داده‌ها
df = pd.read_excel(file_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by='date')

# حذف ردیف‌های خالی در سنسورهای حیاتی
df_clean = df.dropna(subset=all_features).copy()
raw_data = df_clean[all_features].values

# ۳. نرمال‌سازی داده‌ها (Min-Max Scaling)
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(raw_data)

# ۴. ساخت معماری Autoencoder (بازسازی چندمتغیره)
input_dim = len(all_features)
input_layer = Input(shape=(input_dim,))

# Encoder: استخراج ویژگی‌های سیستماتیک
encoded = Dense(16, activation='relu')(input_layer)
latent = Dense(8, activation='relu')(encoded)

# Decoder: بازسازی سیگنال‌ها
decoded = Dense(16, activation='relu')(latent)
output_layer = Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

# ۵. آموزش مدل روی تمام داده‌ها (یادگیری رفتار نرمال سیستم)
print("🚀 مرحله ۱: یادگیری رفتار سیستماتیک سنسورها...")
autoencoder.fit(scaled_data, scaled_data, epochs=150, batch_size=32, shuffle=True, verbose=0)

# ۶. تولید باقیمانده‌ها (Residuals) و شاخص انحراف
reconstructed = autoencoder.predict(scaled_data)
# محاسبه MSE برای هر لحظه (شاخص خام ناهنجاری)
mse_errors = np.mean(np.power(scaled_data - reconstructed, 2), axis=1)
df_clean['Raw_Anomaly_Index'] = mse_errors

# ۷. ارزیابی پیشرفته باقیمانده‌ها (Advanced Residual Evaluation)

# الف) فیلتر EWMA برای شناسایی "روند" و حذف نویز لحظه‌ای
# alpha=0.1 باعث می‌شود تغییرات تدریجی خرابی بهتر دیده شود
df_clean['Smooth_Deviation_Index'] = df_clean['Raw_Anomaly_Index'].ewm(alpha=0.1).mean()

# ب) تحلیل آماری برای تعیین آستانه‌های پویا (Dynamic Thresholds)
# استفاده از صدک‌های داده‌های تاریخی برای قضاوت
p95_threshold = df_clean['Smooth_Deviation_Index'].quantile(0.95)
p99_threshold = df_clean['Smooth_Deviation_Index'].quantile(0.99)

# ج) منطق تصمیم‌گیری (قضاوت نهایی)
def final_judgment(val):
    if val > p99_threshold:
        return "Critical (Red) - Systematic Failure"
    elif val > p95_threshold:
        return "Warning (Yellow) - Operational Drift"
    else:
        return "Normal (Green)"

df_clean['Final_Status'] = df_clean['Smooth_Deviation_Index'].apply(final_judgment)


# ۸. فیلتر کردن خروجی برای "یک ماه آخر"
last_date = df_clean['date'].max()
start_of_last_month = last_date - pd.Timedelta(days=30)
df_output = df_clean[df_clean['date'] >= start_of_last_month].copy()

# ۹. ذخیره خروجی نهایی
os.makedirs(os.path.dirname(output_filename), exist_ok=True)
df_output.to_excel(output_filename, index=False)

print("-" * 50)
print(f"✅ تحلیل هوشمند با موفقیت به پایان رسید.")
print(f"📈 آستانه هشدار (زرد): {p95_threshold:.6f}")
print(f"🚫 آستانه بحرانی (قرمز): {p99_threshold:.6f}")
print(f"📊 بازه زمانی خروجی: {df_output['date'].min()} تا {df_output['date'].max()}")
print(f"📂 فایل خروجی در مسیر زیر ذخیره شد:\n{output_filename}")

🚀 مرحله ۱: یادگیری رفتار سیستماتیک سنسورها...
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
--------------------------------------------------
✅ تحلیل هوشمند با موفقیت به پایان رسید.
📈 آستانه هشدار (زرد): 0.001257
🚫 آستانه بحرانی (قرمز): 0.002904
📊 بازه زمانی خروجی: 2026-05-01 20:50:26 تا 2026-05-31 20:30:17
📂 فایل خروجی در مسیر زیر ذخیره شد:
outputs\G11\dsas_g11_generator_bearings_deviation_monitoring\deviation_monitoring\dsas_g11_generator_bearings_deviation_monitoring_output5.xlsx
